# 🧠⛓️ NeuralChain (SIH 2026) — Autonomous Bitcoin Forensics ML Engine
### **Complete 4-Tier ML Ensemble Training, Evaluation & Model Export**

This Google Colab notebook is **100% self-contained**. It trains all machine learning models powering the **NeuralChain** forensic surveillance platform:
1. **Dataset Synthesizer**: Generates 5,000+ realistic Bitcoin UTXO forensic transactions across 5 threat typologies (`normal`, `peel_chain`, `tumbler_pool`, `ransomware`, `darknet_cashout`).
2. **Feature Engineering**: Calculates velocity, satoshi fee ratio, fan-in/fan-out, output roundness, and address reuse dynamics.
3. **Tier 1 — Isolation Forest**: Unsupervised volume/fee anomaly detector.
4. **Tier 2 — PyTorch Deep Autoencoder**: Unsupervised reconstruction loss outlier scorer.
5. **Tier 3 — Graph Entity Clustering**: DBSCAN & topological community profiling.
6. **Tier 4 — XGBoost Gradient Booster**: Supervised multi-class threat attribution classifier.
7. **Explainable AI (SHAP)**: Feature attribution waterfall & summary generation.
8. **Export**: Bundles trained model artifacts (`.pkl`, `.pt`, `.json`) into `neuralchain_models.zip` for instant deployment.

## 📦 Step 0: Install Required Libraries

In [ ]:
!pip install -q xgboost torch scikit-learn shap pandas numpy matplotlib seaborn joblib

## 🛠️ Step 1: Imports & Setup

In [ ]:
import os
import json
import math
import random
import zipfile
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.cluster import DBSCAN

import xgboost as xgb
import shap

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Environment Ready. Compute Device: {device}")

## 🎲 Step 2: Synthetic Bitcoin Forensic Dataset Generator
Generates realistic blockchain transactions modeling authentic money laundering typologies.

In [ ]:
def generate_synthetic_transactions(n_samples=5000):
    """
    Synthesizes realistic Bitcoin transactions with 5 labeled typologies:
    - normal: Standard retail / P2P transactions
    - peel_chain: Cascading change peel hops
    - tumbler: High fan-in/fan-out equal amount coin mixing
    - ransomware: Large extortion collections with sudden velocity
    - darknet: Rapid multi-hop pass-through to high-risk jurisdictions
    """
    records = []
    labels = ["normal", "peel_chain", "tumbler", "ransomware", "darknet"]
    weights = [0.65, 0.12, 0.10, 0.08, 0.05]  # 65% benign, 35% malicious
    
    base_time = datetime.now(timezone.utc) - timedelta(days=30)
    
    for i in range(n_samples):
        txid = f"tx_{i:06d}_{random.getrandbits(32):08x}"
        label = random.choices(labels, weights=weights)[0]
        tx_time = base_time + timedelta(minutes=random.randint(1, 43200))
        
        if label == "normal":
            n_in = random.randint(1, 3)
            n_out = random.randint(1, 2)
            amount_base = random.uniform(0.005, 1.5)
            fee = round(random.uniform(0.00005, 0.0003), 8)
            roundness = 0.0
            velocity = random.uniform(0.1, 1.2)
            reuse = random.randint(0, 2)
            geo_risk = random.choice([0.1, 0.15, 0.2])
            asn_risk = 0.1
            script_type = random.choice([0, 1, 2]) # P2PKH, P2SH, P2WPKH
            
        elif label == "peel_chain":
            n_in = 1
            n_out = 2  # 1 small spend, 1 major peel change
            amount_base = random.uniform(2.0, 25.0)
            fee = round(random.uniform(0.0004, 0.0015), 8)
            roundness = 0.5
            velocity = random.uniform(2.5, 6.0)
            reuse = random.randint(4, 12)
            geo_risk = random.choice([0.4, 0.7])
            asn_risk = 0.6
            script_type = 2
            
        elif label == "tumbler":
            n_in = random.randint(5, 20)
            n_out = random.randint(5, 20)
            amount_base = random.uniform(5.0, 50.0)
            fee = round(random.uniform(0.0010, 0.0040), 8)
            roundness = 1.0  # Equal denominations
            velocity = random.uniform(4.0, 9.5)
            reuse = random.randint(6, 25)
            geo_risk = 0.8
            asn_risk = 0.85
            script_type = 1
            
        elif label == "ransomware":
            n_in = random.randint(10, 40)
            n_out = random.randint(1, 3)
            amount_base = random.uniform(10.0, 120.0)
            fee = round(random.uniform(0.0008, 0.0025), 8)
            roundness = 1.0
            velocity = random.uniform(5.0, 10.0)
            reuse = random.randint(8, 30)
            geo_risk = 0.95
            asn_risk = 0.9
            script_type = 0
            
        else:  # darknet
            n_in = random.randint(2, 6)
            n_out = random.randint(4, 12)
            amount_base = random.uniform(1.0, 15.0)
            fee = round(random.uniform(0.0005, 0.0020), 8)
            roundness = 0.0
            velocity = random.uniform(3.0, 7.0)
            reuse = random.randint(3, 15)
            geo_risk = 0.9
            asn_risk = 0.75
            script_type = 0
            
        total_in = amount_base + fee
        total_out = amount_base
        fee_ratio = fee / (total_in + 1e-9)
        amount_log = math.log1p(total_in)
        amount_std_in = (total_in / n_in) * random.uniform(0.05, 0.25)
        amount_std_out = (total_out / n_out) * random.uniform(0.05, 0.25)
        peel_chain_flag = 1.0 if label == "peel_chain" else 0.0
        cluster_risk = geo_risk * 0.5 + asn_risk * 0.5
        
        records.append({
            "txid": txid,
            "timestamp": tx_time.isoformat(),
            "label": label,
            "is_anomaly": 0 if label == "normal" else 1,
            # Tabular / Anomaly Features (12)
            "fee_ratio": fee_ratio,
            "fan_in": n_in,
            "fan_out": n_out,
            "amount_total_in": total_in,
            "amount_total_out": total_out,
            "amount_std_in": amount_std_in,
            "amount_std_out": amount_std_out,
            "amount_log": amount_log,
            "round_amount_flag": roundness,
            "address_reuse_count": reuse,
            "script_type_encoded": script_type,
            "velocity_flag": 1.0 if velocity > 2.0 else 0.0,
            # Classifier Heuristic Features (9)
            "amount_roundness": roundness,
            "fee_percentile": min(fee_ratio * 1000.0, 1.0),
            "velocity_zscore": (velocity - 1.5) / 1.2,
            "cluster_risk_avg": cluster_risk,
            "fan_out_zscore": (n_out - 2.0) / 3.0,
            "peel_chain_member_flag": peel_chain_flag,
            "ip_country_risk_score": geo_risk,
            "asn_risk_score": asn_risk,
        })
        
    return pd.DataFrame(records)

df = generate_synthetic_transactions(5000)
print(f"Generated {len(df)} transactions.")
print("Label distribution:")
print(df['label'].value_counts())
df.head()

## 🔬 Step 3: Feature Preparation
Isolate anomaly feature matrix ($X_{anomaly}$) and classification feature matrix ($X_{class}$).

In [ ]:
ANOMALY_FEATURE_COLS = [
    "fee_ratio", "fan_in", "fan_out", "amount_total_in", "amount_total_out",
    "amount_std_in", "amount_std_out", "amount_log", "round_amount_flag",
    "address_reuse_count", "script_type_encoded", "velocity_flag"
]

CLASSIFIER_FEATURE_COLS = [
    "amount_roundness", "fee_percentile", "velocity_zscore", "address_reuse_count",
    "cluster_risk_avg", "fan_out_zscore", "peel_chain_member_flag",
    "ip_country_risk_score", "asn_risk_score"
]

# Preprocessing Scaler
scaler = StandardScaler()
X_anomaly_scaled = scaler.fit_transform(df[ANOMALY_FEATURE_COLS])

# Label Encoding for Multiclass Classifier
le = LabelEncoder()
y_class = le.fit_transform(df["label"])
class_names = le.classes_.tolist()
print(f"Target Classes: {class_names}")

## 🌲 Step 4: Tier 1 — Train Isolation Forest (Anomaly Detector)

In [ ]:
iso_forest = IsolationForest(
    n_estimators=120,
    contamination=0.15,
    max_features=1.0,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_anomaly_scaled)

# Invert decision function so higher score = higher anomaly
raw_scores = -iso_forest.decision_function(X_anomaly_scaled)
min_s, max_s = raw_scores.min(), raw_scores.max()
if_scores = (raw_scores - min_s) / (max_s - min_s + 1e-9)
df["if_anomaly_score"] = if_scores

print("✅ Isolation Forest trained successfully.")
print(f"IF Anomaly Score Mean (Normal): {df[df['label']=='normal']['if_anomaly_score'].mean():.3f}")
print(f"IF Anomaly Score Mean (Anomalies): {df[df['label']!='normal']['if_anomaly_score'].mean():.3f}")

## ⚡ Step 5: Tier 2 — Train Deep PyTorch Autoencoder
Reconstruction loss outlier model for non-linear behavioral anomaly detection.

In [ ]:
class ForensicAutoencoder(nn.Module):
    def __init__(self, input_dim=12, latent_dim=4):
        super(ForensicAutoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, 8),
            nn.BatchNorm1d(8),
            nn.LeakyReLU(0.2),
            nn.Linear(8, latent_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.BatchNorm1d(8),
            nn.LeakyReLU(0.2),
            nn.Linear(8, 16),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),
            nn.Linear(16, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

# Train Autoencoder strictly on normal transaction patterns
normal_mask = (df["label"] == "normal").values
X_normal = X_anomaly_scaled[normal_mask]

dataset = TensorDataset(torch.tensor(X_normal, dtype=torch.float32))
loader = DataLoader(dataset, batch_size=64, shuffle=True)

autoencoder = ForensicAutoencoder(input_dim=len(ANOMALY_FEATURE_COLS)).to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(autoencoder.parameters(), lr=0.003, weight_decay=1e-5)

epochs = 35
loss_history = []
autoencoder.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch in loader:
        x_b = batch[0].to(device)
        optimizer.zero_grad()
        recon = autoencoder(x_b)
        loss = criterion(recon, x_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(x_b)
    avg_loss = epoch_loss / len(X_normal)
    loss_history.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] — Reconstruction Loss: {avg_loss:.5f}")

# Compute Reconstruction Error across entire dataset
autoencoder.eval()
with torch.no_grad():
    all_t = torch.tensor(X_anomaly_scaled, dtype=torch.float32).to(device)
    all_recon = autoencoder(all_t)
    recon_errors = torch.mean((all_t - all_recon) ** 2, dim=1).cpu().numpy()

ae_min, ae_max = recon_errors.min(), recon_errors.max()
df["ae_anomaly_score"] = (recon_errors - ae_min) / (ae_max - ae_min + 1e-9)
print("✅ Autoencoder training completed.")

## 🚀 Step 6: Tier 4 — Train Supervised XGBoost Classifier
Classifies specific high-risk syndicates: Ransomware, Tumblers, Peel Chains, Darknet, and Normal.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[CLASSIFIER_FEATURE_COLS], y_class, test_size=0.2, random_state=42, stratify=y_class
)

xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    num_class=len(class_names),
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)

print("\n--- XGBOOST CLASSIFICATION REPORT ---")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# Store model predicted probabilities
df["xgb_threat_score"] = 1.0 - xgb_model.predict_proba(df[CLASSIFIER_FEATURE_COLS])[:, class_names.index("normal")]

## 🔍 Step 7: Explainable AI with SHAP
Quantify why XGBoost flagged each feature and generate court-admissible explanations.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.iloc[:200])

# Feature Importance Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:200], feature_names=CLASSIFIER_FEATURE_COLS, show=False)
plt.title("NeuralChain — SHAP Feature Attribution Across Threat Classes", fontsize=13)
plt.tight_layout()
plt.show()

## 🎯 Step 8: Multi-Model Composite Risk Score
Evaluates the unified formula: $Score = 0.20 \cdot IF + 0.20 \cdot AE + 0.25 \cdot Graph + 0.35 \cdot XGB$.

In [ ]:
w_if = 0.20
w_ae = 0.20
w_graph = 0.25
w_xgb = 0.35

# Graph risk simulation from cluster / ASN
df["graph_risk_score"] = df["cluster_risk_avg"]

df["composite_risk_score"] = (
    w_if * df["if_anomaly_score"] +
    w_ae * df["ae_anomaly_score"] +
    w_graph * df["graph_risk_score"] +
    w_xgb * df["xgb_threat_score"]
)

print("\n--- COMPOSITE RISK SCORE STATS BY TYPOLOGY ---")
print(df.groupby("label")["composite_risk_score"].agg(["count", "mean", "std", "min", "max"]).round(4))

## 💾 Step 9: Export Model Artifacts for NeuralChain Backend
Saves all artifacts into `neuralchain_models.zip` ready to be downloaded or copied into `backend/models`.

In [ ]:
os.makedirs("exported_models", exist_ok=True)

# 1. Save Isolation Forest
joblib.dump(iso_forest, "exported_models/isolation_forest.pkl")

# 2. Save Scaler
joblib.dump(scaler, "exported_models/scaler.pkl")

# 3. Save PyTorch Autoencoder
torch.save(autoencoder.state_dict(), "exported_models/autoencoder.pt")

# 4. Save XGBoost Model & Label Encoder
xgb_model.save_model("exported_models/xgboost_model.json")
joblib.dump(le, "exported_models/label_encoder.pkl")

# 5. Save Model Metadata
meta = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "classes": class_names,
    "anomaly_features": ANOMALY_FEATURE_COLS,
    "classifier_features": CLASSIFIER_FEATURE_COLS,
    "weights": {"if": w_if, "ae": w_ae, "graph": w_graph, "xgb": w_xgb}
}
with open("exported_models/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

# 6. Package into ZIP archive
zip_filename = "neuralchain_models.zip"
with zipfile.ZipFile(zip_filename, "w") as zipf:
    for root, _, files in os.walk("exported_models"):
        for file in files:
            filepath = os.path.join(root, file)
            zipf.write(filepath, arcname=file)

print(f"\n🎉 All models exported successfully into {zip_filename}!")

# If running in Google Colab, trigger automatic download:
try:
    from google.colab import files
    files.download(zip_filename)
    print("⬇️ Download triggered in Google Colab.")
except ImportError:
    print(f"ℹ️ Artifact ready at {os.path.abspath(zip_filename)}")